<a href="https://colab.research.google.com/github/Shambhaviadhikari/PythonClass/blob/main/Titanic_Survival_Classifier_(Fall_2024)_%5BG37903602%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instructions




Perform classification using the Titanic dataset. Compare the performance of different classifier models, including Logistic Regression, Random Forest Classifier, and XGBoost Classifier.





Incorporate the following aspects:

Part 1 - Data Processing

+ **Load the data**.
+ **Explore the data**, including distributions, correlation, etc. Make plots.
+ Check for null values. **Handle null values** by dropping or imputing. How will you handle the "age" variable?
+ **Choose target and feature(s)**.
+ **Encode features** as necessary (ordinal vs one-hot).
+ **Scale / normalize features** as necessary.
+ **Split into train and test sets** (specifically 80/20 split). Remember to use a random seed to ensure your results are reproducible.

Part 2 - Benchmark Models

A) Logistic Regression:

+ Use a [`LogisticRegression` model](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) from `sklearn` (using default params for now).
+ **Train the model** using the training data.
+ Inspect artifacts from the training process:
    + **Print the model's coefficients**. Inspect the coefficients by wrapping them in a pandas Series (or DataFrame for multiclass) and labeling them with their corresponding feature names, then sort them in descending order.
    + **Interpret the coefficients** - which features contribute most to our model's predictive ability?
+ Make **predictions** for the test set.
+ Evaluate the results using sklearn **classification metrics**, specifically the classification_report function. Optionally also report on the ROC AUC score as an additional metric. Interpret the results - how well did the model do?
+ Make a **confusion matrix**, using the `confusion_matrix` function from `sklearn.metrics`. Display your confusion matrix as a heatmap, for example using plotly or matplotlib.

B) Random Forest:

+ Repeat Step 2, but this time using a [`RandomForestClassifier` model](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) from `sklearn` (using default params for now).

C) XGBoost:
  + Repeat Step 2, but this time using an [`XGBClassifier` model](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier) from `xgboost` (using default params for now).

Part 3 - Best Model

  + Compare the results from Steps 2, 3, and 4. Which model gives the best benchmark performance?
  + Try researching and using different classification models to improve performance, if possible.
  + Try engineering different features to improve the model's performance, if possible.
  + Try tuning the model(s) hyperparameters to improve the model's performance, if possible.
  + Optionally use a [`GridSearchCV`](https://scikit-learn.org/dev/modules/generated/sklearn.model_selection.GridSearchCV.html) from `sklearn` to explore the hyperparameter space to find the hyperparameters that yield the best performance for each model. Specifically use ROC AUC score ("roc_auc" for binary classification or "roc_auc_ovr" for multi-class classification) as the metric / scoring function to optimize on.

  + What is the best model you found? Report on it's  performance. What are the hyperparameters of the best model?



## Part 1 - Data Processing


### Data Loading

We are loading a combined version of the Titanic dataset that has been hosted on GitHub.

For more information about this dataset, consult: https://www.kaggle.com/competitions/titanic/data

In [ ]:
from pandas import read_csv

df = read_csv("https://raw.githubusercontent.com/prof-rossetti/intro-to-python/main/data/titanic-original-full.csv")
df["embarked"] = df["embarked"].str.upper()
df.head()

,passenger_id,survived,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,False,3,"Braund, Mr. Owen Harris",MALE,22.0,1,0,A/5 21171,7.2500,NaN,SOUTHAMPTON
1,2,True,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",FEMALE,38.0,1,0,PC 17599,71.2833,C85,CHERBOURG
2,3,True,3,"Heikkinen, Miss. Laina",FEMALE,26.0,0,0,STON/O2. 3101282,7.9250,NaN,SOUTHAMPTON
3,4,True,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",FEMALE,35.0,1,0,113803,53.1000,C123,SOUTHAMPTON
4,5,False,3,"Allen, Mr. William Henry",MALE,35.0,0,0,373450,8.0500,NaN,SOUTHAMPTON


### Data Exploration

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   passenger_id  1309 non-null   int64  
 1   survived      1309 non-null   bool   
 2   pclass        1309 non-null   int64  
 3   name          1309 non-null   object 
 4   gender        1309 non-null   object 
 5   age           1046 non-null   float64
 6   sibsp         1309 non-null   int64  
 7   parch         1309 non-null   int64  
 8   ticket        1309 non-null   object 
 9   fare          1308 non-null   float64
 10  cabin         295 non-null    object 
 11  embarked      1307 non-null   object 
dtypes: bool(1), float64(2), int64(4), object(5)
memory usage: 113.9+ KB


In [ ]:
df.describe()

,passenger_id,pclass,age,sibsp,parch,fare
count,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,655.000000,2.294882,29.881138,0.498854,0.385027,33.295479
std,378.020061,0.837836,14.413493,1.041658,0.865560,51.758668
min,1.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,328.000000,2.000000,21.000000,0.000000,0.000000,7.895800
50%,655.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,982.000000,3.000000,39.000000,1.000000,0.000000,31.275000
max,1309.000000,3.000000,80.000000,8.000000,9.000000,512.329200


In [ ]:
import plotly.express as px
# Survival distribution
survival_fig = px.histogram(df, x="survived", title="Survival Distribution", labels={"survived": "Survived"})
survival_fig.update_layout(width=600, height=400)  # Set custom width and height
survival_fig.show()

# Age distribution
age_fig = px.histogram(df, x="age", title="Age Distribution", labels={"age": "Age"}, nbins=10)
age_fig.update_layout(width=600, height=400)  # Set custom width and height
age_fig.show()

# Fare distribution
fare_fig = px.histogram(df, x="fare", title="Fare Distribution", labels={"fare": "Fare"}, nbins=10)
fare_fig.update_layout(width=600, height=400)  # Set custom width and height
fare_fig.show()

# Passenger class vs survival
pclass_survival_fig = px.histogram(df, x="pclass", color="survived",
                                   title="Passenger Class vs Survival",
                                   labels={"pclass": "Passenger Class", "survived": "Survived"})
pclass_survival_fig.update_layout(width=600, height=400)  # Set custom width and height
pclass_survival_fig.show()


In [ ]:
import plotly.graph_objects as go

# Select only numeric columns for correlation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
correlation_matrix = df[numeric_cols].corr()

# Plot the heatmap
# The 'df' argument was replaced with 'data' and assigned the heatmap trace
heatmap = go.Figure(
    data=[go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.index,
        colorscale='RdBu',  # Replace 'coolwarm' with a supported colorscale
        colorbar=dict(title="Correlation")
    )]
)
heatmap.update_layout(
    title="Correlation Heatmap",
    xaxis=dict(title="Features"),
    yaxis=dict(title="Features"),
    width=600,
    height=400
)
heatmap.show()

The correlation heatmap shows that features like age, fare, and family-related attributes (sibsp, parch) are somewhat correlated, while passenger_id and pclass show minimal correlation with other features.

Null value


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)


In [ ]:
df['age'] = df.groupby('pclass')['age'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
import pandas as pd

# Step 1: Calculate the average age for each Pclass
average_age = df.groupby('pclass')['age'].mean()

# Step 2: Define the impute_age function
def impute_age(row):
    """
    Imputes the age based on the Pclass average.

    Args:
        row: A row of the DataFrame.

    Returns:
        The imputed age.
    """
    age = row['age']
    pclass = row['pclass']

    if pd.isnull(age):
        return average_age[pclass]  # Use the calculated average for that Pclass
    else:
        return age

# Step 3: Apply the function to the DataFrame
df['age'] = df.apply(impute_age, axis=1)



We calculated the average age for each Pclass and used it to impute missing age values in the DataFrame based on the passenger's Pclass.

In [ ]:
# Fill missing 'embarked' values with the most common value
df['embarked'] = df['embarked'].fillna('S')

# 2. Feature Engineering
# Creating a new feature 'family_size' by combining 'sibsp' and 'parch'
df['family_size'] = df['sibsp'] + df['parch']


In [ ]:
# 3. Drop Columns
# Dropping 'cabin', 'name', atd 'Ticket' columns as they are not useful
df.drop(['cabin', 'name', 'ticket'], axis=1, inplace=True)

In [ ]:
# Fill missing fare with median value (or mean if you prefer)
df['fare'] = df['fare'].fillna(df['fare'].median())

In [ ]:
# 3. Encode categorical features
gender = pd.get_dummies(df['gender'], drop_first=True)
embark = pd.get_dummies(df['embarked'], drop_first=True)

In [ ]:
# Drop the original 'gender' and 'embarked' columns
df.drop(['gender', 'embarked'], axis=1, inplace=True)

# Concatenate the new dummy columns to the dataframe
df = pd.concat([df, gender, embark], axis=1)

In [ ]:
# 5. Drop Rows with Any Remaining Missing Values
df.dropna(inplace=True)

In [ ]:
df

,passenger_id,survived,pclass,age,sibsp,parch,fare,family_size,MALE,QUEENSTOWN,S,SOUTHAMPTON
0,1,False,3,22.000000,1,0,7.2500,1,True,False,False,True
1,2,True,1,38.000000,1,0,71.2833,1,False,False,False,False
2,3,True,3,26.000000,0,0,7.9250,0,False,False,False,True
3,4,True,1,35.000000,1,0,53.1000,1,False,False,False,True
4,5,False,3,35.000000,0,0,8.0500,0,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,True,3,24.816367,0,0,8.0500,0,True,False,False,True
1305,1306,True,1,39.000000,0,0,108.9000,0,False,False,False,False
1306,1307,True,3,38.500000,0,0,7.2500,0,True,False,False,True
1307,1308,True,3,24.816367,0,0,8.0500,0,True,False,False,True


In [ ]:
# Import the necessary module
from sklearn.preprocessing import StandardScaler

# 6. Scaling Numeric Features
numerical_features = ['age', 'fare', 'family_size']  # Include any other numerical features as needed
scaler = StandardScaler() # Now, StandardScaler is recognized
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# 7. Handle Class Imbalance with SMOTE
X = df.drop(['survived'], axis=1)
y = df['survived']

In [ ]:
# Import the necessary module
from sklearn.model_selection import train_test_split

# 8. Split Data into Train and Test Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Part 2 - Benchmark Models

### A) - Logistic Regression


In [ ]:
# Import necessary libraries
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import plotly.figure_factory as ff

# Train the Logistic Regression model
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)

# Predictions
log_pred = log_model.predict(X_test)

In [ ]:
# Evaluate performance
print("Logistic Regression Report:\n", classification_report(y_test, log_pred))
print("ROC AUC Score:", roc_auc_score(y_test, log_model.predict_proba(X_test)[:, 1]))

Logistic Regression Report:
               precision    recall  f1-score   support

       False       0.76      0.79      0.77        99
        True       0.87      0.85      0.86       163

    accuracy                           0.82       262
   macro avg       0.81      0.82      0.81       262
weighted avg       0.83      0.82      0.83       262

ROC AUC Score: 0.8676333891057817


The Logistic Regression model achieves 82% accuracy with a strong ROC AUC score of 0.87, indicating good performance in distinguishing survivors and non-survivors.

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, log_pred)

# Plot the confusion matrix using Plotly
fig = ff.create_annotated_heatmap(
    z=conf_matrix,
    x=['Predicted No', 'Predicted Yes'],
    y=['Actual No', 'Actual Yes'],
    colorscale='Viridis',
    showscale=True
)

fig.update_layout(title="Confusion Matrix - Logistic Regression")
fig.show()

### B) Random Forest

In [ ]:
# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier

# 9. Train the Random Forest Model
rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# 10. Predictions and Evaluation
rf_pred = rf.predict(X_test)

In [ ]:
# Classification Report
print("Random Forest Classification Report:\n", classification_report(y_test, rf_pred))

Random Forest Classification Report:
               precision    recall  f1-score   support

       False       0.84      0.86      0.85        99
        True       0.91      0.90      0.91       163

    accuracy                           0.89       262
   macro avg       0.88      0.88      0.88       262
weighted avg       0.89      0.89      0.89       262



In [ ]:
# ROC AUC Score
print("ROC AUC Score:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

ROC AUC Score: 0.9330420772138562


The Random Forest model performs well with 89% accuracy and an excellent ROC AUC score of 0.93, indicating strong classification capability for both survivors and non-survivors.

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, rf_pred)

# Plot the confusion matrix using Plotly
fig = ff.create_annotated_heatmap(
    z=conf_matrix,
    x=['Predicted No', 'Predicted Yes'],
    y=['Actual No', 'Actual Yes'],
    colorscale='Blues',
    showscale=True
)
fig.update_layout(title="Confusion Matrix - Random Forest")
fig.show()

### C) XGBoost

In [ ]:
%%capture
!pip install xgboost

In [ ]:
# Import necessary libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import plotly.figure_factory as ff
from xgboost import XGBClassifier

# Train the XGBoost model
xgb_model = XGBClassifier(n_estimators=500, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning:

[22:04:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.




XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [ ]:
# Predictions
xgb_pred = xgb_model.predict(X_test)

In [ ]:
# Evaluate performance
print("XGBoost Classification Report:\n", classification_report(y_test, xgb_pred))
print("ROC AUC Score:", roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]))

XGBoost Classification Report:
               precision    recall  f1-score   support

       False       0.80      0.82      0.81        99
        True       0.89      0.88      0.88       163

    accuracy                           0.85       262
   macro avg       0.85      0.85      0.85       262
weighted avg       0.86      0.85      0.86       262

ROC AUC Score: 0.9268141538080189


The XGBoost model achieves 85% accuracy and a high ROC AUC score of 0.93, demonstrating robust performance with balanced precision and recall for predicting survivors and non-survivors.

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, xgb_pred)

# Plot the confusion matrix using Plotly
fig = ff.create_annotated_heatmap(
    z=conf_matrix,
    x=['Predicted No', 'Predicted Yes'],
    y=['Actual No', 'Actual Yes'],
    colorscale='Viridis',
    showscale=True
)

fig.update_layout(title="Confusion Matrix - XGBoost")
fig.show()

## Part 3 - Best Model

In [ ]:
# Compare model performances
model_performance = {
    "Logistic Regression": roc_auc_score(y_test, log_model.predict_proba(X_test)[:, 1]),
    "Random Forest": roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]),
    "XGBoost": roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
}

best_model = max(model_performance, key=model_performance.get)
print(f"Best Model: {best_model} with ROC AUC: {model_performance[best_model]}")


Best Model: Random Forest with ROC AUC: 0.9330420772138562


The best-performing model is the **Random Forest**, achieving the highest ROC AUC score of **0.933**, indicating superior ability in distinguishing between survivors and non-survivors.